# Analise - Comunicacao convergente (chat -> voz)

Notebook da etapa **Analisar** do GPSC. Le `metrics.csv` (gerado pelo chat-server) e produz os graficos do experimento.

- **Objetivo 2:** tempo de estabelecimento e duracao das chamadas.
- **Objetivo 4:** comportamento sob degradacao de rede (tc/netem, coluna `tag`).
- **Objetivo 3:** banda texto x voz (a partir de `banda.csv`, preenchido com dados do Wireshark).

> Se `metrics.csv` ainda nao existir, o notebook gera dados de EXEMPLO so para voce ver a forma dos graficos.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = None
for p in ['metrics.csv', 'metrics/metrics.csv', '../metrics/metrics.csv']:
    if os.path.exists(p):
        CSV = p
        break

if CSV:
    df = pd.read_csv(CSV)
    print(f'Dados reais: {len(df)} chamadas  ({CSV})')
else:
    print('metrics.csv nao encontrado - gerando dados de EXEMPLO.')
    rng = np.random.default_rng(42)
    base = {'baseline': 400, 'delay100': 650, 'loss5': 950, 'loss10': 1600}
    rows = []
    for tag, mu in base.items():
        for _ in range(30):
            rows.append({'timestamp': '', 'tag': tag,
                         'setup_ms': max(50, rng.normal(mu, mu * 0.2)),
                         'duration_s': max(1, rng.normal(45, 15))})
    df = pd.DataFrame(rows)

df.head()

In [ ]:
# Resumo do tempo de estabelecimento por perfil de rede (tag)
resumo = df.groupby('tag')['setup_ms'].agg(['count', 'mean', 'median', 'std']).round(1)
resumo

In [ ]:
# Objetivo 4: tempo de estabelecimento medio por perfil
ORDER = ['baseline', 'delay50', 'delay100', 'delay200', 'loss1', 'loss3', 'loss5', 'loss10']
tags = [t for t in ORDER if t in df['tag'].unique()] or sorted(df['tag'].unique())
means = df.groupby('tag')['setup_ms'].mean().reindex(tags)

plt.figure(figsize=(7, 4))
means.plot(kind='bar', color='steelblue')
plt.ylabel('Tempo de estabelecimento (ms)')
plt.xlabel('Perfil de rede')
plt.title('Tempo de estabelecimento da chamada por perfil')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao do tempo de estabelecimento por perfil
plt.figure(figsize=(7, 4))
df.boxplot(column='setup_ms', by='tag', grid=False)
plt.suptitle('')
plt.title('Distribuicao do tempo de estabelecimento por perfil')
plt.xlabel('Perfil de rede')
plt.ylabel('Tempo de estabelecimento (ms)')
plt.tight_layout()
plt.show()

In [ ]:
# Objetivo 2: distribuicao da duracao das chamadas
plt.figure(figsize=(7, 4))
df['duration_s'].plot(kind='hist', bins=15, color='seagreen', edgecolor='white')
plt.xlabel('Duracao da chamada (s)')
plt.title('Distribuicao da duracao das chamadas')
plt.tight_layout()
plt.show()

## Banda: texto x voz (Objetivo 3)

Preencha `banda.csv` com os valores medidos no Wireshark (kbps), uma linha por canal:

```
canal,kbps
texto,2
voz,84
```

In [ ]:
BANDA = 'banda.csv'
if os.path.exists(BANDA):
    b = pd.read_csv(BANDA)
else:
    print('banda.csv nao encontrado - usando valores ILUSTRATIVOS.')
    b = pd.DataFrame({'canal': ['texto (WebSocket)', 'voz (RTP/DTLS)'], 'kbps': [2, 84]})

plt.figure(figsize=(5, 4))
plt.bar(b['canal'], b['kbps'], color=['#2563eb', '#16a34a'])
plt.ylabel('Banda (kbps)')
plt.title('Banda consumida por canal')
plt.tight_layout()
plt.show()
b

## Interpretacao

_(Preencher com os dados reais apos as execucoes.)_

- O tempo de estabelecimento cresce com atraso/perda; o **limiar de inviabilidade** observado foi em torno de `...`.
- A voz consome cerca de `...x` mais banda que o texto.
- Conclusao sobre o custo da convergencia: `...`.